<a href="https://colab.research.google.com/github/Adnan0425ninja/Student-Mark-Analyzer/blob/main/Student_Mark_Analyzer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Student Mark Analyzer
**Project Owner: Md Adnan Ashraf**

This notebook analyzes student marks with Pandas, NumPy, Matplotlib and Seaborn.

> The included CSV is synthetic sample data created for this educational project.


In [ ]:
# Install dependencies (run once in Google Colab if needed)
!pip -q install pandas numpy matplotlib seaborn


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")
SUBJECTS = ["Mathematics", "Programming", "Physics"]
PASS_MARK = 40


## 1. Load the dataset
Upload `student_marks.csv` in Colab when prompted, or update the path for local use.

In [ ]:
# Google Colab upload option
try:
    from google.colab import files
    uploaded = files.upload()
    csv_file = next(iter(uploaded))
except ImportError:
    csv_file = "../data/student_marks.csv"

df = pd.read_csv(csv_file)
display(df.head())
print("Dataset shape:", df.shape)


## 2. Explore and validate the data

In [ ]:
print("Columns:", df.columns.tolist())
print("\\nMissing values:")
display(df.isna().sum().to_frame("Missing values"))
print("\\nDataset information:")
df.info()


In [ ]:
required = ["Student_ID", "Student_Name", *SUBJECTS]
missing = [col for col in required if col not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

for subject in SUBJECTS:
    df[subject] = pd.to_numeric(df[subject], errors="coerce")

if df[SUBJECTS].isna().any().any():
    raise ValueError("Subject marks contain missing or non-numeric values.")
if ((df[SUBJECTS] < 0) | (df[SUBJECTS] > 100)).any().any():
    raise ValueError("Marks must be between 0 and 100.")


## 3. Calculate total, average, result and grade

In [ ]:
df["Total_Marks"] = df[SUBJECTS].sum(axis=1)
df["Average_Marks"] = df[SUBJECTS].mean(axis=1).round(2)
df["Result"] = np.where(df[SUBJECTS].min(axis=1) >= PASS_MARK, "Pass", "Fail")
df["Grade"] = pd.cut(
    df["Average_Marks"],
    bins=[-0.01, 49.99, 59.99, 69.99, 79.99, 89.99, 100],
    labels=["F", "D", "C", "B", "A", "A+"]
).astype(str)
display(df.head(10))


## 4. Class performance summary

In [ ]:
print("Total students:", len(df))
print("Class average:", round(df["Average_Marks"].mean(), 2))
print("Highest average:", df["Average_Marks"].max())
print("Lowest average:", df["Average_Marks"].min())
print("Pass count:", (df["Result"] == "Pass").sum())
print("Fail count:", (df["Result"] == "Fail").sum())
print("Pass percentage:", round((df["Result"] == "Pass").mean() * 100, 2), "%")
print("\\nSubject-wise averages:")
display(df[SUBJECTS].mean().round(2).to_frame("Average Marks"))


## 5. Top five students

In [ ]:
top_five = df.nlargest(5, "Average_Marks")[
    ["Student_ID", "Student_Name", "Total_Marks", "Average_Marks", "Grade"]
]
display(top_five)


## 6. Visualizations

In [ ]:
# Average marks by subject
plt.figure(figsize=(8, 5))
df[SUBJECTS].mean().plot(kind="bar")
plt.title("Average Marks by Subject")
plt.ylabel("Average marks (out of 100)")
plt.ylim(0, 100)
plt.tight_layout()
plt.show()


In [ ]:
# Pass/fail distribution
plt.figure(figsize=(6, 5))
df["Result"].value_counts().reindex(["Pass", "Fail"], fill_value=0).plot(
    kind="pie", autopct="%1.1f%%", startangle=90
)
plt.title("Pass / Fail Distribution")
plt.ylabel("")
plt.tight_layout()
plt.show()


In [ ]:
# Grade distribution
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x="Grade", order=["F", "D", "C", "B", "A", "A+"])
plt.title("Grade Distribution")
plt.xlabel("Grade")
plt.ylabel("Number of students")
plt.tight_layout()
plt.show()


In [ ]:
# Top ten students
top_ten = df.nlargest(10, "Average_Marks").sort_values("Average_Marks")
plt.figure(figsize=(9, 6))
top_ten.plot.barh(x="Student_Name", y="Average_Marks", legend=False)
plt.title("Top 10 Students by Average Marks")
plt.xlabel("Average marks")
plt.xlim(0, 100)
plt.tight_layout()
plt.show()


## 7. Export analyzed results

In [ ]:
output_file = "analyzed_student_results.csv"
df.to_csv(output_file, index=False)
print("Saved:", output_file)

# In Google Colab, uncomment to download:
# from google.colab import files
# files.download(output_file)
